In [ ]:
import gc

import numpy as np
from pyparsing import line
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score as sk_f1_score
from scipy import stats
from scipy.stats import linregress
import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt

import confidence_functions as cf

BASE_DIR = '/gpfs/projects/b1042/AmaralLab/shin/DNN-Decision-Uncertainty-Quantification/models'
CACHE_PATH = '/gpfs/projects/b1042/AmaralLab/shin/DNN-Decision-Uncertainty-Quantification/cached_data_224.pt'
MODEL_TAG = 'resnet18_224x224_8ep'
NUM_CLASSES = 19
CIFAR_ROOT = '/gpfs/projects/b1042/AmaralLab/shin/DNN-Decision-Uncertainty-Quantification/data'
WORKERS = 4  # for DataLoader num_workers


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# DataLoader generator
g = torch.Generator()
g.manual_seed(SEED)

In [ ]:
def compute_per_sample_f1(labels, preds, eps=1e-12):
    """
    Manual per-sample (a.k.a. 'samples'-averaged, but kept per-row) F1.
    sklearn's f1_score(..., average='samples') only returns one aggregate
    number across the dataset, not one value per sample -- this gives the
    per-row values that aggregate reduces over.
    Rows with no true positives AND no predicted positives get F1=0.0,
    matching sklearn's zero_division=0 convention (flip to 1.0 if you'd
    rather treat "correctly predicted nothing" as a perfect match).
    """
    labels = labels.astype(np.float32)
    preds = preds.astype(np.float32)

    tp = (preds * labels).sum(axis=1)
    fp = (preds * (1 - labels)).sum(axis=1)
    fn = ((1 - preds) * labels).sum(axis=1)

    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, (2 * tp) / (denom + eps), 0.0)
    return f1

In [ ]:
def scatter_plot(x, y, xlabel, ylabel, title, save_path, color='steelblue'):
    # get rrid of nan and inf in x and y
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # print if there were any NaN or inf values removed
    n_removed = len(mask) - np.sum(mask)
    print(f"Removed {n_removed} samples with NaN or inf values from scatter plot.")

    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    line = slope * x + intercept
    print(f"Scatter plot: {title}")
    print(f"Slope: {slope:.2e}, R-value: {r_value:.2f}, R2: {r_value**2:.2f}, P-value: {p_value:.2e}")

    plt.figure(figsize=(6, 5))
    plt.scatter(x, y, s=8, alpha=0.4, color=color)
    plt.plot(x, line, color="red", linewidth=2, label="Line of Best Fit")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_plot_three_groups(x, y, n_test, n_adv, n_ood,
                               xlabel, ylabel, title, save_path):
    # get rrid of nan and inf in x and y
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # print if there were any NaN or inf values removed
    n_removed = len(mask) - np.sum(mask)
    print(f"Removed {n_removed} samples with NaN or inf values from scatter plot.")

    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    line = slope * x + intercept
    print(f"Scatter plot: {title}")
    print(f"Slope: {slope:.4f}, R-value: {r_value:.4f}, R2: {r_value**2:.2f}, P-value: {p_value:.4f}")

    x_test, y_test = x[:n_test], y[:n_test]
    x_adv, y_adv = x[n_test:n_test + n_adv], y[n_test:n_test + n_adv]
    x_ood, y_ood = x[n_test + n_adv:], y[n_test + n_adv:]

    plt.figure(figsize=(10, 5))
    plt.scatter(x_ood, y_ood, s=10, alpha=0.6, color='green', label='OOD (out-of-distribution)')
    plt.scatter(x_adv, y_adv, s=10, alpha=0.6, color='blue', label='Adversarial')
    plt.scatter(x_test, y_test, s=10, alpha=0.6, color='gray', label='Test (in-distribution)')

    plt.plot(x, line, color="red", linewidth=2, label="Line of Best Fit")

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
def hexbin_plot_three_groups(x, y, n_test, n_adv, n_ood,
                               xlabel, ylabel, title, save_path):
    # get rrid of nan and inf in x and y
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # print if there were any NaN or inf values removed
    n_removed = len(mask) - np.sum(mask)
    print(f"Removed {n_removed} samples with NaN or inf values from scatter plot.")

    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    line = slope * x + intercept
    print(f"Scatter plot: {title}")
    print(f"Slope: {slope:.4f}, R-value: {r_value:.4f}, R2: {r_value**2:.2f}, P-value: {p_value:.4f}")

    x_test, y_test = x[:n_test], y[:n_test]
    x_adv, y_adv = x[n_test:n_test + n_adv], y[n_test:n_test + n_adv]
    x_ood, y_ood = x[n_test + n_adv:], y[n_test + n_adv:]

    plt.figure(figsize=(10, 5))
    plt.hexbin(x_ood, y_ood, bins=10, cmap='Blues', label='OOD (out-of-distribution)')
    plt.hexbin(x_adv, y_adv, bins=10, cmap='Blues', label='Adversarial')
    plt.hexbin(x_test, y_test, bins=10, cmap='Blues', label='Test (in-distribution)')

    plt.plot(x, line, color="red", linewidth=2, label="Line of Best Fit")

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# make box plot for in distribution vs vs adversarial vs out of distribution for mean above threshold prob, stability, and mahalanobis
def box_plot_three_groups(x, n_test, n_adv, n_ood, xlabel, title, save_path):
    # get rrid of nan and inf in x
    mask = np.isfinite(x)
    x = x[mask]

    # print if there were any NaN or inf values removed
    n_removed = len(mask) - np.sum(mask)
    print(f"Removed {n_removed} samples with NaN or inf values from box plot.")

    plt.figure(figsize=(6, 5))
    x_test, x_adv, x_ood = x[:n_test], x[n_test:n_test + n_adv], x[n_test + n_adv:]

    plt.boxplot([x_test, x_adv, x_ood], tick_labels=['Test (in-distribution)', 'Adversarial', 'OOD (out-of-distribution)'])
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [1]:
prob = torch.load(f'{BASE_DIR}/sigmoid_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['prob']
test_labels = torch.load(f'{BASE_DIR}/test_labels_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['test_labels']
stability = torch.load(f'{BASE_DIR}/stability_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['stability']
threshold = np.asarray(torch.load(f'{BASE_DIR}/threshold_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['threshold'])
stats = torch.load(f'{BASE_DIR}/stats_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['stats']

prob_np = prob.numpy() if torch.is_tensor(prob) else np.asarray(prob)
labels_np = test_labels.numpy() if torch.is_tensor(test_labels) else np.asarray(test_labels)
stability_np = stability.numpy() if torch.is_tensor(stability) else np.asarray(stability)

preds_np = (prob_np > threshold).astype(np.float32)
f1_per_sample = compute_per_sample_f1(labels_np, preds_np)


NameError: name 'torch' is not defined

In [ ]:
# mean sigmoid prob among classes ABOVE threshold, per sample
above_thresh_mask = prob_np > threshold
mean_prob_above = np.full(len(prob_np), np.nan)
for i in range(len(prob_np)):
    vals = prob_np[i][above_thresh_mask[i]]
    if len(vals) > 0:
        mean_prob_above[i] = vals.mean()

In [ ]:
scatter_plot(
    f1_per_sample, mean_prob_above,
    xlabel="Per-sample F1", ylabel="Mean Sigmoid Prob Above Threshold",
    title="F1 vs Sigmoid Prob per Sample",
    save_path=f'{BASE_DIR}/f1_vs_mean_prob.png'
)

scatter_plot(
    f1_per_sample, stability_np,
    xlabel="Per-sample F1", ylabel="Mean Shannon Entropy Across",
    title="F1 vs Entropy per Sample",
    save_path=f'{BASE_DIR}/f1_vs_shannon.png'
)

In [ ]:
del prob, test_labels, stability, preds_np, above_thresh_mask, mean_prob_above
gc.collect() 

In [ ]:
alpha = torch.load(f'{BASE_DIR}/alpha_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['alpha']
alpha

In [ ]:
print(stats)

In [ ]:
f1_combined = torch.load(f'{BASE_DIR}/f1_combined_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['f1_combined']
mean_prob_combined = torch.load(f'{BASE_DIR}/mean_prob_combined_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['mean_prob_combined']
stability_combined_np = torch.load(f'{BASE_DIR}/stability_combined_np_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['stability_combined_np']

In [ ]:
maha_combined_np = torch.load(f'{BASE_DIR}/maha_combined_np_{MODEL_TAG}.pt', map_location='cpu', weights_only=False)['maha_combined_np']
maha_combined_np

In [ ]:
scatter_plot_three_groups(
    f1_combined, mean_prob_combined, n_test, n_adv, n_ood,
    xlabel="F1 per Sample", ylabel="Mean Sigmoid Prob Above Threshold",
    title="F1 vs Sigmoid Prob",
    save_path=f'{BASE_DIR}/f1_vs_mean_prob_combined.png'
)

scatter_plot_three_groups(
    f1_combined, stability_combined_np, n_test, n_adv, n_ood,
    xlabel="F1 per Sample", ylabel="Entropy from Sigmoid Prob",
    title="F1 vs Entropy",
    save_path=f'{BASE_DIR}/f1_vs_stability_combined.png'
)

scatter_plot_three_groups(
    f1_combined, maha_combined_np, n_test, n_adv, n_ood,
    xlabel="F1 per Sample", ylabel="Mahalanobis Confidence",
    title="F1 vs Mahalanobis Confidence",
    save_path=f'{BASE_DIR}/f1_vs_mahalanobis_combined.png'
)

In [ ]:
box_plot_three_groups(
    mean_prob_combined, n_test, n_adv, n_ood,
    xlabel="Sigmoid Probabilities",
    title="Mean Sigmoid Probability Above Threshold",
    save_path=f'{BASE_DIR}/box_mean_prob_combined.png'
)

box_plot_three_groups(
    stability_combined_np, n_test, n_adv, n_ood,
    xlabel="Shannon Entropy",
    title="Entropy using Sigmoid Prob",
    save_path=f'{BASE_DIR}/box_stability_combined.png'
)

box_plot_three_groups(
    maha_combined_np, n_test, n_adv, n_ood,
    xlabel="Confidence Score",
    title="Mahalanobis Confidence",
    save_path=f'{BASE_DIR}/box_mahalanobis_combined.png'
)

In [ ]:
import pandas as pd
import seaborn as sns

# Helper function to remove NaN/Inf and slice groups
def clean_and_split(x, n_test, n_adv, n_ood):
    mask = np.isfinite(x)
    x = x[mask]
    return x[:n_test], x[n_test:n_test + n_adv], x[n_test + n_adv:]

group_labels = (['Test (in-distribution)'] * n_test + 
                ['Adversarial'] * n_adv + 
                ['OOD (out-of-distribution)'] * n_ood)

p_test, p_adv, p_ood = clean_and_split(mean_prob_combined, n_test, n_adv, n_ood)
s_test, s_adv, s_ood = clean_and_split(stability_combined_np, n_test, n_adv, n_ood)
m_test, m_adv, m_ood = clean_and_split(maha_combined_np, n_test, n_adv, n_ood)

df = pd.DataFrame({
    'Group': group_labels,
    'Mean Probability': np.concatenate([p_test, p_adv, p_ood]),
    'Shannon Stability': np.concatenate([s_test, s_adv, s_ood]),
    'Mahalanobis Confidence': np.concatenate([m_test, m_adv, m_ood])
})

# Reshape into Tidy Format
df_melted = pd.melt(df, id_vars=['Group'], var_name='Metric', value_name='Value')

palette = {'Test (in-distribution)': '#2b5c8f', 'Adversarial': '#d95f02', 'OOD (out-of-distribution)': '#7570b3'}

g = sns.catplot(
    data=df_melted,
    x='Methods',
    y='Confidence',
    col='Metric',
    hue='Data Group',
    kind='box',
    palette=palette,
    sharey=False,  # Allows independent Y-scales per metric
    height=4,
    aspect=1.0,
    dodge=False
)

g.set_xticklabels(rotation=15)
g.set_titles(col_template="{col_name}")
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/combined_box_plots_subplots.png', dpi=150)



# everything on a single axis for easier comparison of distributions across metrics
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_melted,
    x='Metric',
    y='Value',
    hue='Group',
    palette=['#2b5c8f', '#d95f02', '#7570b3']
)

plt.title('Distribution Comparison across Metrics', fontsize=24)
plt.xlabel('Metric', fontsize=18)
plt.ylabel('Value', fontsize=18)
plt.legend(title='Dataset Group', frameon=True, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()